# 5G Network Intelligence — Zero-Inflated Forecasting

## Question and hypothesis

Can explicit zero probability reduce original-unit MAE relative to direct L1? This is a research hypothesis, not a promised improvement. Fixed 40 causal inputs; no external source is added.

## Temporal protocol

Fit/stopping/scoring boundaries: 335/383/443 and 395/443/503. Stopping selects component rounds only. Score windows cannot fit thresholds, probabilities or quantiles. The windows have previously supported research, so scores remain exploratory. Validation 504–671 and final holdout are excluded.


## Configuration and execution control

Default review does not train. Only an explicit hosted execution may enable training. All checkpoints remain in Kaggle Output.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = 'be432db077c36c15aa8cbbc9fe52cd4bf72d5cb3'


## Environment and inherited source

Use the completed private multisource worker output as a reproducible source/data attachment. No credentials or network clone is needed. Only relevant new implementation appears below.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train_multisource_pilot.py') if (p.parent.parent / 'data/processed/panel.npz').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one canonical multisource output')
    PROJECT = Path('/kaggle/working/hurdle_research')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src', 'scripts', 'configs', 'data/processed'):
        shutil.copytree(roots[0]/folder, PROJECT/folder, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)


## Fixed protocol

Four positive conditional quantiles and one active-probability classifier. No class reweighting or tuned gate: probability must retain its probabilistic interpretation.


In [ ]:
# Fixed protocol: changing these values creates a new experiment.
PROTOCOL = {'protocol': 'hurdle-development-v1',
 'horizons': [1, 6, 24],
 'folds': [{'name': 'inner1', 'fit_end': 335, 'stop_end': 383, 'score_end': 443},
           {'name': 'inner2', 'fit_end': 395, 'stop_end': 443, 'score_end': 503}],
 'conditional_quantiles': [0.05, 0.1, 0.25, 0.5],
 'max_rounds': 800,
 'patience': 60,
 'stride': 1,
 'seed': 17,
 'threads': 4,
 'params': {'num_leaves': 63,
            'learning_rate': 0.035,
            'min_data_in_leaf': 100,
            'lambda_l2': 0.5},
 'threshold': 'fixed_mixture_median_not_tuned',
 'probability_calibration': 'none',
 'inputs': 'original_40_causal_features',
 'reused_validation_504_671': 'forbidden_in_this_experiment',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden'}
if RUN_TRAINING:
    import yaml
    Path('configs/hurdle_research.yaml').write_text(yaml.safe_dump(PROTOCOL), encoding='utf-8')


## Model: mixture median

For active probability p≤0.5, predict zero. Otherwise use positive conditional quantile q=1−0.5/p. This follows the mixture CDF at its median. Interpolation and sorted crossing repair approximate that quantile; probability miscalibration can hurt MAE.


In [ ]:
"""Pure inference for a zero-mass plus positive-quantile mixture."""

import numpy as np


def mixture_median(active_probability, positive_quantiles, levels):
    """Approximate mixture median; no probability calibration or learned thresholds.

    For P(Y>0)=p>0.5, the positive conditional quantile is 1-0.5/p.
    Otherwise zero is a median. Interpolation includes the fixed nonnegative
    lower boundary (0,0); sparse quantiles and crossing repair are approximations.
    """
    p = np.asarray(active_probability, dtype=float)
    values = np.asarray(positive_quantiles, dtype=float)
    levels = np.asarray(levels, dtype=float)
    if (p.ndim != 1 or values.shape != (len(p), len(levels))
            or levels.ndim != 1 or not len(levels)
            or not np.isfinite(levels).all() or levels[0] <= 0
            or levels[-1] != 0.5 or not (np.diff(levels) > 0).all()):
        raise ValueError("Expected ordered conditional levels ending at 0.5")
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any() or not np.isfinite(values).all():
        raise ValueError("Finite probabilities in [0,1] and conditional predictions required")
    q = np.zeros_like(p)
    active = p > 0.5
    q[active] = 1 - 0.5 / p[active]
    grid = np.r_[0, levels]
    repaired = np.column_stack([np.zeros(len(p)), np.sort(np.maximum(values, 0), axis=1)])
    upper = np.searchsorted(grid, q, side="right").clip(1, len(grid) - 1)
    lower = upper - 1
    fraction = (q - grid[lower]) / (grid[upper] - grid[lower])
    index = np.arange(len(p))
    result = repaired[index, lower] * (1 - fraction) + repaired[index, upper] * fraction
    return np.where(active, result, 0)


## Training implementation

The attached MIT source contains the cloud trainer, inference helper and fixed YAML. The trainer keeps fit, early stopping and score arrays separate. It saves per-component rounds, checkpoints and predictions; it never selects on scoring outcomes.


In [ ]:
if RUN_TRAINING:
    patches = list(Path('/kaggle/input').rglob('train_hurdle_research.py'))
    if len(patches) != 1:
        raise RuntimeError('Attach exactly one hurdle source dataset')
    patch = patches[0].parent
    shutil.copyfile(patch/'train_hurdle_research.py', 'scripts/train_hurdle_research.py')
    shutil.copyfile(patch/'hurdle.py', 'src/models/hurdle.py')


## Hosted training and debugging

One classifier, four positive quantile models and one direct L1 control per fold/horizon. Checkpoints are saved after each completed fold/horizon. Failures preserve logs and partial output. LightGBM binary/quantile objectives follow [official parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html).


In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm>=4.6,<5', 'numpy', 'pandas', 'pyyaml'], check=True)
    with Path('reports/hurdle_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.train_hurdle_research'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    with tarfile.open('/kaggle/working/hurdle_results.tar.gz', 'w:gz') as archive:
        for folder in ('reports', 'configs', 'artifacts/hurdle_research'):
            if Path(folder).exists():
                archive.add(folder, arcname=folder)
    if status:
        raise RuntimeError(f'Cloud fitting failed: {status}; see saved log')
else:
    print('Review mode. No fitting, calibration or holdout evaluation.')


## Saved evaluation and limitations

MAE uses original units; WAPE is a ratio on the full observed scoring population. Report every fold/horizon and zero false-positive rate. A lower zero false-positive rate alone does not prove improvement. No probability calibration or unseen-test claim.


In [ ]:
path = Path('reports/hurdle_development_scores.csv')
if path.exists():
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed scores attached.')
